In [1]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [3]:
import os

print(os.listdir())

['ml_env', 'Untitled1.ipynb', '.ipynb_checkpoints', 'parsers', 'soc_env', 'ml_pipeline', 'autoencoder.ipynb', 'Untitled.ipynb', 'datasets', 'data', 'capstone_demo.ipynb']


In [4]:
import numpy as np

data = np.load("ml_pipeline/window_features.npy")

print("Shape:", data.shape)

Shape: (21176, 312)


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

In [6]:
feature_idx = 0

original = data[:, feature_idx]
scaled = data_scaled[:, feature_idx]

print("Original mean:", original.mean())
print("Scaled mean:", scaled.mean())

print("Original std:", original.std())
print("Scaled std:", scaled.std())

Original mean: 10630.601688231962
Scaled mean: 9.81391855662609e-15
Original std: 5843.190073527653
Scaled std: 1.0000000000000018


In [7]:
import joblib

joblib.dump(scaler, "ml_pipeline/scaler.pkl")


['ml_pipeline/scaler.pkl']

In [8]:
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self, input_dim=312):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, 32)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            
            nn.Linear(64, 128),
            nn.ReLU(),
            
            nn.Linear(128, input_dim)
        )
    
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [9]:
model = Autoencoder().to(device)
print(model)

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=312, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=312, bias=True)
  )
)


In [10]:
from torch.utils.data import DataLoader, TensorDataset

In [11]:
criterion = nn.MSELoss()

In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)

In [13]:
epochs = 30
losses = []

In [14]:
import torch

tensor_data = torch.tensor(data_scaled, dtype=torch.float32)

In [15]:
from torch.utils.data import TensorDataset
dataset = TensorDataset(tensor_data)

In [16]:
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

In [17]:
for batch in dataloader:
    print(batch[0].shape)
    break

torch.Size([256, 312])


In [18]:
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in dataloader:
        x = batch[0].to(device)
        output = model(x)
        loss = criterion(output, x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    losses.append(avg_loss)

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.6f}")

Epoch [1/30] Loss: 0.640192
Epoch [2/30] Loss: 0.442603
Epoch [3/30] Loss: 0.368064
Epoch [4/30] Loss: 0.322307
Epoch [5/30] Loss: 0.288533
Epoch [6/30] Loss: 0.264110
Epoch [7/30] Loss: 0.248199
Epoch [8/30] Loss: 0.234913
Epoch [9/30] Loss: 0.228704
Epoch [10/30] Loss: 0.222281
Epoch [11/30] Loss: 0.212670
Epoch [12/30] Loss: 0.205549
Epoch [13/30] Loss: 0.196168
Epoch [14/30] Loss: 0.190078
Epoch [15/30] Loss: 0.185083
Epoch [16/30] Loss: 0.182515
Epoch [17/30] Loss: 0.178862
Epoch [18/30] Loss: 0.176624
Epoch [19/30] Loss: 0.170461
Epoch [20/30] Loss: 0.164868
Epoch [21/30] Loss: 0.160829
Epoch [22/30] Loss: 0.160156
Epoch [23/30] Loss: 0.154509
Epoch [24/30] Loss: 0.155647
Epoch [25/30] Loss: 0.168716
Epoch [26/30] Loss: 0.147572
Epoch [27/30] Loss: 0.142907
Epoch [28/30] Loss: 0.143462
Epoch [29/30] Loss: 0.140453
Epoch [30/30] Loss: 0.138919


In [19]:
model.eval()

with torch.no_grad():
    data_tensor = tensor_data.to(device)
    reconstructions = model(data_tensor)

    errors = torch.mean((data_tensor - reconstructions) ** 2, dim=1)
    errors = errors.cpu().numpy()

In [32]:
upper_bound = np.percentile(errors_log, 99.9)  # less aggressive

errors_log_scaled = np.where(
    errors_log > upper_bound,
    upper_bound + np.log1p(errors_log - upper_bound),  # soft compression
    errors_log
)
median_log = np.median(errors_log_scaled)
mad_log = np.median(np.abs(errors_log_scaled - median_log)) + 1e-6

severity_scores = (errors_log_scaled - median_log) / mad_log

/tmp/ipykernel_3012/1048864259.py:5: RuntimeWarning: invalid value encountered in log1p
  upper_bound + np.log1p(errors_log - upper_bound),  # soft compression


In [31]:
print("Sample severity scores:", severity_scores[:20])
print("Min severity:", np.min(severity_scores))
print("Max severity:", np.max(severity_scores))

Sample severity scores: [21.734991  21.734991  21.734991  21.734991  21.734991  21.734991
 21.734991  21.734991  12.830768  10.967915   9.553446  11.326733
 13.696741  10.007986  10.900498   6.1656857  7.760597   8.574525
 12.280021  14.85415  ]
Min severity: -2.4036672
Max severity: 21.734991


In [20]:
for p in [90, 92, 95, 97, 99]:
    thresh = np.percentile(errors, p)
    anomalies = errors > thresh
    print(f"{p}% → {np.sum(anomalies)} anomalies")

90% → 2118 anomalies
92% → 1694 anomalies
95% → 1059 anomalies
97% → 636 anomalies
99% → 212 anomalies


In [23]:
median = np.median(errors)
mad = np.median(np.abs(errors - median))  # Median Absolute Deviation

threshold = median + 6 * mad  
print("Median:", median)
print("MAD:", mad)
print("New Threshold:", threshold)
print("Total samples:", len(errors))
print("Anomalies detected:", np.sum(anomalies))

Median: 0.068587415
MAD: 0.025073372
New Threshold: 0.21902764
Total samples: 21176
Anomalies detected: 212
